# RTX-OOM-Guard — Colab T4 Validation

**Goal:** Measure whether proactive defragmentation delays or prevents OOM during a Transformer training loop with deliberate memory fragmentation.

**Hardware:** NVIDIA T4 (free Colab), 15GB VRAM  
**Runtime:** GPU → T4  
**Expected time:** ~5 minutes

## Instructions
1. Open in Colab (Runtime → Change runtime type → T4 GPU)
2. Run all cells
3. Results saved to `results/colab_t4_results.json`

In [ ]:
# Cell 1: Setup
!git clone https://github.com/poojakira/RTX-OOM-Guard.git /content/RTX-OOM-Guard 2>/dev/null
!pip install -e /content/RTX-OOM-Guard --quiet
!pip install torch --quiet

In [ ]:
# Cell 2: Verify GPU
import torch
import gc
import json
import time

assert torch.cuda.is_available(), "No GPU — change runtime to T4"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Cell 3: Fragmentation helper
import random

def fragment_memory(n_tensors=200, sizes_mb=(1, 4, 16, 64)):
    """Allocate and free tensors of varying sizes to fragment CUDA caching allocator."""
    tensors = []
    for _ in range(n_tensors):
        size_mb = random.choice(sizes_mb)
        t = torch.randn(size_mb * 256 * 1024, device='cuda')
        tensors.append(t)
    # Free every other tensor — creates holes
    for i in range(0, len(tensors), 2):
        tensors[i] = None
    tensors = [t for t in tensors if t is not None]
    gc.collect()
    torch.cuda.empty_cache()
    return tensors  # keep refs so holes persist

In [ ]:
# Cell 4: Baseline — train WITHOUT guard
def train_baseline(max_steps=300):
    torch.cuda.reset_peak_memory_stats()
    model = torch.nn.Transformer(d_model=512, nhead=8, num_encoder_layers=6).cuda()
    optimizer = torch.optim.Adam(model.parameters())
    
    frags = fragment_memory()
    
    steps_completed = 0
    t0 = time.time()
    try:
        for step in range(max_steps):
            src = torch.randn(32, 64, 512, device='cuda')
            tgt = torch.randn(32, 64, 512, device='cuda')
            out = model(src, tgt)
            loss = out.sum()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            steps_completed = step + 1
            del src, tgt, out, loss
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            pass
        else:
            raise
    
    elapsed = time.time() - t0
    stats = torch.cuda.memory_stats()
    result = {
        "steps": steps_completed,
        "peak_gb": round(stats['allocated_bytes.all.peak'] / 1e9, 3),
        "retries": stats.get('num_alloc_retries', 0),
        "oom": steps_completed < max_steps,
        "elapsed_s": round(elapsed, 1)
    }
    
    del model, optimizer, frags
    gc.collect()
    torch.cuda.empty_cache()
    return result

print("=== BASELINE (no guard) ===")
baseline = train_baseline()
for k, v in baseline.items():
    print(f"  {k}: {v}")

In [ ]:
# Cell 5: WITH guard
from rtx_oom_guard import auto_instrument

def train_with_guard(max_steps=300):
    torch.cuda.reset_peak_memory_stats()
    model = torch.nn.Transformer(d_model=512, nhead=8, num_encoder_layers=6).cuda()
    optimizer = torch.optim.Adam(model.parameters())
    model, optimizer = auto_instrument(model, optimizer)
    
    frags = fragment_memory()
    
    steps_completed = 0
    t0 = time.time()
    try:
        for step in range(max_steps):
            src = torch.randn(32, 64, 512, device='cuda')
            tgt = torch.randn(32, 64, 512, device='cuda')
            out = model(src, tgt)
            loss = out.sum()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            steps_completed = step + 1
            del src, tgt, out, loss
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            pass
        else:
            raise
    
    elapsed = time.time() - t0
    stats = torch.cuda.memory_stats()
    result = {
        "steps": steps_completed,
        "peak_gb": round(stats['allocated_bytes.all.peak'] / 1e9, 3),
        "retries": stats.get('num_alloc_retries', 0),
        "oom": steps_completed < max_steps,
        "elapsed_s": round(elapsed, 1)
    }
    
    del model, optimizer, frags
    gc.collect()
    torch.cuda.empty_cache()
    return result

print("\n=== WITH RTX-OOM-GUARD ===")
guarded = train_with_guard()
for k, v in guarded.items():
    print(f"  {k}: {v}")

In [ ]:
# Cell 6: Summary and save
results = {
    "gpu": torch.cuda.get_device_name(0),
    "vram_gb": round(torch.cuda.get_device_properties(0).total_mem / 1e9, 1),
    "baseline": baseline,
    "guarded": guarded,
    "improvement": {
        "extra_steps": guarded['steps'] - baseline['steps'],
        "peak_reduction_gb": round(baseline['peak_gb'] - guarded['peak_gb'], 3),
        "oom_prevented": baseline['oom'] and not guarded['oom']
    }
}

print("\n" + "="*50)
print("VALIDATION RESULTS")
print("="*50)
print(json.dumps(results, indent=2))

# Save
import os
os.makedirs('/content/RTX-OOM-Guard/results', exist_ok=True)
with open('/content/RTX-OOM-Guard/results/colab_t4_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("\nSaved to results/colab_t4_results.json")

## Interpreting Results

| Outcome | What it means |
|---------|---------------|
| Guard prevents OOM | Compaction freed enough fragmented blocks to survive |
| Guard delays OOM (more steps) | Partial win — compaction helps but optimizer state still fragments |
| Guard doesn't help | Optimizer state (Adam exp_avg/exp_avg_sq) dominates fragmentation and isn't compacted |

**Any result is valid.** A documented negative result with root-cause analysis is engineering. Fabricated positive results are fraud.